# Preprocesamiento de datos — Capa Bronze

Análisis exploratorio sobre los datos crudos ingestados en `nessie.bronze` para
justificar las reglas de limpieza aplicadas en silver. Se cubren cuatro bloques por
tabla: tipos de datos, datos faltantes, outliers y valores inválidos. Una última
sección resume cada regla silver con la evidencia cuantitativa encontrada en bronze.

Tablas analizadas: `nessie.bronze.yellow_taxi` y `nessie.bronze.taxi_zone_lookup`.

In [ ]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie

spark = get_spark("preprocess-bronze")
nessie.set_spark_branch(spark, "bronze")
spark.version

## 1. Yellow Taxi

In [ ]:
yt = spark.table("nessie.bronze.yellow_taxi").cache()
total_yt = yt.count()
print(f"Total de registros: {total_yt:,}")
print(f"Columnas: {len(yt.columns)}")

### 1.1 Análisis de tipos de datos

La ingesta bronze unifica el *schema drift* del origen NYC TLC: entre años los
parquet originales alternan `INT`/`BIGINT` en los IDs, `INT`/`DOUBLE` en
`passenger_count` y `RatecodeID`, y renombran `Airport_fee` → `airport_fee` en 2025.
Bronze los normaliza a un esquema canónico con *casts* explícitos; aquí verificamos
el resultado.

In [ ]:
# Esquema canonical resultante tras la ingesta
yt.printSchema()

In [ ]:
# Histórico de evolución del esquema (Iceberg)
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.history").show(truncate=False)

In [ ]:
# cbd_congestion_fee (añadida en 2025) solo debería tener valores desde ese año.
# Lo validamos contando no-nulos por año.
from pyspark.sql.functions import col, year as spark_year, sum as spark_sum, when

yt.groupBy(spark_year(col("tpep_pickup_datetime")).alias("year")) \
  .agg(
      spark_sum(when(col("cbd_congestion_fee").isNotNull(), 1).otherwise(0)).alias("cbd_no_null"),
      spark_sum(when(col("airport_fee").isNotNull(), 1).otherwise(0)).alias("airport_fee_no_null"),
      spark_sum(when(col("congestion_surcharge").isNotNull(), 1).otherwise(0)).alias("congestion_no_null"),
  ) \
  .orderBy("year") \
  .show(50)

### 1.2 Análisis de datos faltantes

Distinguimos dos tipos de nulos:
- **Estructurales**: columnas que no existían en determinados años (p.ej.
  `congestion_surcharge` desde 2019, `cbd_congestion_fee` desde 2025). No son
  errores y no se filtran en silver.
- **Anómalos**: nulos en columnas clave (`tpep_pickup_datetime`,
  `tpep_dropoff_datetime`) que impiden cualquier análisis temporal. Silver los
  elimina.

In [ ]:
# Nulos totales por columna
null_counts = yt.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in yt.columns
])
null_counts.show(vertical=True)

In [ ]:
# Porcentaje de nulos (sólo columnas con al menos un nulo)
row = null_counts.collect()[0].asDict()
pct = [(c, n, round(100 * n / total_yt, 2)) for c, n in row.items() if n > 0]
pct.sort(key=lambda t: -t[1])
for c, n, p in pct:
    print(f"  {c:<25s}  {n:>10,}  ({p:5.2f}%)")

In [ ]:
# Nulos por año en columnas con nulos estructurales vs anómalos.
# Confirma que congestion_surcharge y cbd_congestion_fee son estructurales
# (nulos concentrados en años anteriores a su introducción).
nulls_year = yt.groupBy(spark_year(col("tpep_pickup_datetime")).alias("year")).agg(
    spark_sum(col("passenger_count").isNull().cast("int")).alias("passenger_count_null"),
    spark_sum(col("RatecodeID").isNull().cast("int")).alias("RatecodeID_null"),
    spark_sum(col("store_and_fwd_flag").isNull().cast("int")).alias("store_and_fwd_null"),
    spark_sum(col("congestion_surcharge").isNull().cast("int")).alias("congestion_null"),
    spark_sum(col("airport_fee").isNull().cast("int")).alias("airport_fee_null"),
    spark_sum(col("cbd_congestion_fee").isNull().cast("int")).alias("cbd_null"),
).orderBy("year")
nulls_year.show(50)

In [ ]:
# Filas con timestamps ausentes — no recuperables
null_pickup = yt.filter(col("tpep_pickup_datetime").isNull()).count()
null_dropoff = yt.filter(col("tpep_dropoff_datetime").isNull()).count()
print(f"tpep_pickup_datetime NULL:  {null_pickup:,}")
print(f"tpep_dropoff_datetime NULL: {null_dropoff:,}")

### 1.3 Análisis de outliers y valores inválidos

Cada apartado muestra la evidencia detrás de una regla de limpieza silver.

#### 1.3.1 Timestamps fuera de rango

El dataset contiene viajes con fechas imposibles (años 2001, 2038, 2088…). El
recuento mensual en bronze ya los exhibe: provienen de errores en el taxímetro.

In [ ]:
# Distribución de años fuera del rango esperado (2019-2025 de las cargas Q1)
from pyspark.sql.functions import month as spark_month

year_counts = yt.groupBy(
    spark_year(col("tpep_pickup_datetime")).alias("year")
).count().orderBy("year")
year_counts.show(50)

#### 1.3.2 Timestamps inconsistentes (pickup ≥ dropoff)

Viajes cuyo *dropoff* es anterior o igual al *pickup*: imposibles físicamente.

In [ ]:
bad_ts = yt.filter(col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime")).count()
print(f"Filas con pickup >= dropoff: {bad_ts:,}  ({100 * bad_ts / total_yt:.3f}%)")

#### 1.3.3 Importes y distancias negativos

Valores negativos en `trip_distance`, `fare_amount`, `tip_amount`, `tolls_amount`
y `total_amount` no tienen interpretación física en un viaje normal (en el feed
TLC suelen ser ajustes contables y *chargebacks*).

In [ ]:
neg_checks = [
    ("trip_distance", col("trip_distance") < 0),
    ("fare_amount",   col("fare_amount") < 0),
    ("total_amount",  col("total_amount") < 0),
    ("tip_amount",    col("tip_amount") < 0),
    ("tolls_amount",  col("tolls_amount") < 0),
]
for name, cond in neg_checks:
    n = yt.filter(cond).count()
    print(f"  {name:<15s}  negativos: {n:>8,}  ({100 * n / total_yt:.3f}%)")

In [ ]:
# Resumen estadístico: min/max muestran la magnitud del ruido
yt.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "tolls_amount"
).summary("min", "25%", "50%", "75%", "max").show()

#### 1.3.5 IDs de localización fuera de rango

`PULocationID` y `DOLocationID` deben referenciar una zona válida (1–265 según
la tabla de referencia TLC). Los IDs 264 ("Unknown") y 265 ("N/A") se conservan
intencionadamente en silver — descartamos solo lo que está fuera del rango.

In [ ]:
# Percentiles altos: revelan la forma de la cola derecha. Si p99 y max están
# muy separados hay una cola larga de valores atípicos candidata a ser recortada.
yt.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "tolls_amount"
).summary("50%", "90%", "95%", "99%", "99.9%", "max").show()

In [ ]:
#### 1.3.6 Códigos categóricos inválidos

Valores que no figuran en el diccionario de datos TLC 2024 indican
corrupción del taxímetro o registros de prueba.

In [ ]:
# Top 20 por fare_amount — ¿son negociados (RatecodeID=5), viajes al
# aeropuerto o basura? Revisar visualmente decide si el tope superior
# captura ruido o recorta viajes legítimos caros.
yt.select(
    "tpep_pickup_datetime", "trip_distance", "fare_amount", "total_amount",
    "RatecodeID", "payment_type", "PULocationID", "DOLocationID"
).orderBy(col("fare_amount").desc()).limit(20).show(truncate=False)

In [ ]:
# Top 20 por trip_distance — un viaje dentro de NYC rara vez supera ~40 mi
# (JFK → norte de Westchester). Valores de cientos de millas son errores del
# taxímetro; deberían caer fuera de cualquier cota superior razonable.
yt.select(
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
    "fare_amount", "RatecodeID", "PULocationID", "DOLocationID"
).orderBy(col("trip_distance").desc()).limit(20).show(truncate=False)

#### 1.3.7 Passenger count fuera de lo físicamente posible

La ley NYC TLC permite 5 pasajeros en un taxi estándar, más la excepción de
menores de 7 años en el regazo de un adulto → techo razonable = 6. Los valores
7, 8 y 9 que aparecen en bronze son probables *sentinels* o errores del
taxímetro (no caben físicamente en un sedán de 4–5 plazas) y se descartan.

In [ ]:
pu_out = yt.filter(~col("PULocationID").between(1, 265)).count()
do_out = yt.filter(~col("DOLocationID").between(1, 265)).count()
print(f"PULocationID fuera de [1,265]: {pu_out:,}")
print(f"DOLocationID fuera de [1,265]: {do_out:,}")

#### 1.3.5 Códigos categóricos inválidos

Valores que no figuran en el diccionario de datos TLC 2024 indican
corrupción del taxímetro o registros de prueba.

In [ ]:
# Valores presentes vs. valores válidos según diccionario TLC
print("VendorID (válidos 1,2,6,7):")
yt.groupBy("VendorID").count().orderBy("VendorID").show(truncate=False)

print("RatecodeID (válidos 1-6, 99):")
yt.groupBy("RatecodeID").count().orderBy("RatecodeID").show(truncate=False)

print("payment_type (válidos 0-6):")
yt.groupBy("payment_type").count().orderBy("payment_type").show(truncate=False)

print("store_and_fwd_flag (válidos Y, N):")
yt.groupBy("store_and_fwd_flag").count().orderBy("store_and_fwd_flag").show(truncate=False)

In [ ]:
# Recuento agregado de filas con al menos un código inválido
invalid_vendor = yt.filter(~col("VendorID").isin([1, 2, 6, 7])).count()
invalid_rate   = yt.filter(~col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99])).count()
invalid_pay    = yt.filter(~col("payment_type").isin([0, 1, 2, 3, 4, 5, 6])).count()
invalid_flag   = yt.filter(~col("store_and_fwd_flag").isin(["Y", "N"])).count()
print(f"VendorID inválido:            {invalid_vendor:,}")
print(f"RatecodeID inválido:          {invalid_rate:,}")
print(f"payment_type inválido:        {invalid_pay:,}")
print(f"store_and_fwd_flag inválido:  {invalid_flag:,}")

#### 1.3.6 Passenger count fuera de lo físicamente posible

La ley NYC TLC permite 5 pasajeros en un taxi estándar, más la excepción de
menores de 7 años en el regazo de un adulto → techo razonable = 6. Los valores
7, 8 y 9 que aparecen en bronze son probables *sentinels* o errores del
taxímetro (no caben físicamente en un sedán de 4–5 plazas) y se descartan.

In [ ]:
# Distribución completa (incluye nulos y valores extremos)
yt.groupBy("passenger_count").count().orderBy("passenger_count").show(50, truncate=False)

In [ ]:
# Filas fuera del rango [1,6] (incluye 0, 7, 8, 9 y nulos)
out_range = yt.filter(~col("passenger_count").between(1, 6)).count()
zero = yt.filter(col("passenger_count") == 0).count()
print(f"passenger_count fuera de [1,6] o nulo: {out_range:,}  ({100 * out_range / total_yt:.2f}%)")
print(f"  de los cuales passenger_count = 0:   {zero:,}")

#### 1.3.7 Duplicados exactos

Silver aplica `dropDuplicates()` global como red de seguridad (mismas 20
columnas idénticas). Cuantificamos la magnitud aquí.

In [ ]:
distinct_yt = yt.dropDuplicates().count()
dups = total_yt - distinct_yt
print(f"Filas totales:      {total_yt:,}")
print(f"Filas distintas:    {distinct_yt:,}")
print(f"Duplicados exactos: {dups:,}  ({100 * dups / total_yt:.3f}%)")

## 2. Taxi Zone Lookup

In [ ]:
zl = spark.table("nessie.bronze.taxi_zone_lookup").cache()
total_zl = zl.count()
print(f"Total de registros: {total_zl}")
print(f"Columnas: {len(zl.columns)}")

### 2.1 Análisis de tipos de datos

La tabla de referencia llega en CSV y sus tipos son estables. Silver renombra
las columnas a *snake_case* (`LocationID` → `location_id`, `Borough` →
`borough`, …) para homogeneizar el estilo del lakehouse.

In [ ]:
zl.printSchema()

### 2.2 Análisis de datos faltantes

Toda zona debería tener `LocationID`, `Borough` y `Zone`. `service_zone` sí
puede ser nulo en alguna zona administrativa.

In [ ]:
# Nulos por columna
null_counts_zl = zl.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in zl.columns
])
null_counts_zl.show()

In [ ]:
# Strings vacíos o solo whitespace (no detectados como nulos pero igual de problemáticos)
from pyspark.sql.functions import trim, length

empty_borough = zl.filter((length(trim(col("Borough"))) == 0) | col("Borough").isNull()).count()
empty_zone = zl.filter((length(trim(col("Zone"))) == 0) | col("Zone").isNull()).count()
empty_service = zl.filter((length(trim(col("service_zone"))) == 0) | col("service_zone").isNull()).count()
print(f"Borough vacío o nulo:      {empty_borough}")
print(f"Zone vacío o nulo:         {empty_zone}")
print(f"service_zone vacío o nulo: {empty_service}")

### 2.3 Análisis de outliers y valores inválidos

In [ ]:
# LocationID fuera del rango esperado [1, 265]
out_range_zl = zl.filter(~col("LocationID").between(1, 265)).count()
print(f"LocationID fuera de [1,265]: {out_range_zl}")

# Duplicados en la clave
dup_ids = zl.groupBy("LocationID").count().filter(col("count") > 1).count()
print(f"LocationID duplicados:       {dup_ids}")

In [ ]:
# Valores sentinel "Unknown" / "N/A" — se conservan (264, 265 son casos válidos del TLC)
zl.filter(
    col("Borough").isin(["Unknown", "N/A"]) | col("Zone").isin(["Unknown", "N/A"])
).show(truncate=False)

In [ ]:
# Dominio de boroughs (detecta typos o valores inesperados)
zl.groupBy("Borough").count().orderBy(col("count").desc()).show(truncate=False)

## 3. Resumen — reglas silver justificadas

Se recogen en una sola tabla las reglas aplicadas en
`src/etl/silver/clean_yellow_taxi.py` y `clean_zone_lookup.py`, junto al
recuento de filas bronze que cada regla descarta (evidencia cuantitativa).

In [ ]:
# Cada regla aislada contra bronze — útil para el informe final del TFM.
# Nota: las reglas se aplican en cadena en silver; aquí se miden por separado
# (la suma no es la resta total porque una fila puede violar varias).
rules = [
    ("pickup NULL",               col("tpep_pickup_datetime").isNull()),
    ("dropoff NULL",              col("tpep_dropoff_datetime").isNull()),
    ("pickup >= dropoff",         col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime")),
    ("trip_distance < 0",         col("trip_distance") < 0),
    ("fare_amount < 0",           col("fare_amount") < 0),
    ("total_amount < 0",          col("total_amount") < 0),
    ("tip_amount < 0",            col("tip_amount") < 0),
    ("tolls_amount < 0",          col("tolls_amount") < 0),
    ("PULocationID fuera [1,265]",~col("PULocationID").between(1, 265)),
    ("DOLocationID fuera [1,265]",~col("DOLocationID").between(1, 265)),
    ("VendorID inválido",         ~col("VendorID").isin([1, 2, 6, 7])),
    ("RatecodeID inválido",       ~col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99])),
    ("payment_type inválido",     ~col("payment_type").isin([0, 1, 2, 3, 4, 5, 6])),
    ("store_and_fwd_flag inválido", ~col("store_and_fwd_flag").isin(["Y", "N"])),
    ("passenger_count fuera [1,6]", ~col("passenger_count").between(1, 6)),
]

print(f"{'regla':<32s} {'filas':>12s}  {'%':>6s}")
print("-" * 55)
for name, cond in rules:
    n = yt.filter(cond).count()
    print(f"{name:<32s} {n:>12,}  {100 * n / total_yt:>5.2f}%")

In [ ]:
# Liberar caches
yt.unpersist()
zl.unpersist()
spark.stop()